In [40]:
import os
import re
import json
import tempfile
import numpy as np
import fitz
import pdfplumber
import faiss
import google.genai as genai
from google.genai import types
from dotenv import load_dotenv

In [42]:
load_dotenv()

True

In [43]:
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)
 
EMBED_MODEL = "models/gemini-embedding-001"
EMBED_DIM   = 3072       # text-embedding-004 outputs 768-dim vectors
 
CHUNK_SIZE    = 500      # target words per chunk
CHUNK_OVERLAP = 60       # words of overlap between consecutive chunks
MIN_CHUNK_WORDS = 25     # discard chunks shorter than this (tail fragments)

In [3]:
# Regex to detect section headings
_HEADING_RE = re.compile(
    r"(?m)^("
    r"\d+[\.\d]*\s+[A-Z][A-Za-z]"              # "1.2 Safety Instructions"
    r"|[A-Z]{4,}[\sA-Z]*[!:.]?\s*$"            # "MAINTENANCE" / "MAINTENANCE:"
    r"|(?:WARNING|CAUTION|NOTE|IMPORTANT|SAFETY)[!:.]?"  # "WARNING!" / "IMPORTANT:"
    r")",
)

In [4]:
pdf_path = "manual2.pdf"

In [5]:
doc = fitz.open(pdf_path)
pages = []
for page_num, page in enumerate(doc, start=1):
    raw = page.get_text("text", sort=True)

    # Basic noise removal: skip blank pages
    if len(raw.strip()) < 30:
        continue

    pages.append({"page": page_num, "text": raw})

doc.close()

In [6]:
pages[0]

{'page': 1,
 'text': '       80L Electric Oven\n\n                  Model Number STR-5F80L\n\n          INSTRUCTION MANUAL\n\n\n\n\nAFTER SALES SUPPORT                                                                                info@tempo.org\n     AUS  1300 886 649                                                  MODEL: STR-5F80L PRODUCT CODE: 700805 04/2020    E209'}

In [7]:
tables_out = []
 
with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):
        raw_tables = page.extract_tables()

        for tbl in raw_tables:
            if not tbl:
                continue

            # Convert each row to "Col1 | Col2 | Col3" format
            rows = []
            for row in tbl:
                # pdfplumber returns None for empty cells
                cleaned = [str(cell).strip() if cell else "" for cell in row]
                row_text = " | ".join(cleaned)
                if row_text.strip(" |"):   # skip fully empty rows
                    rows.append(row_text)

            if not rows:
                continue

            table_text = f"[TABLE - page {page_num}]\n" + "\n".join(rows)
            tables_out.append({"page": page_num, "text": table_text})

print(f"[step2] Extracted {len(tables_out)} tables")

[step2] Extracted 2 tables


In [8]:
tables_out[0]

{'page': 17,
 'text': '[TABLE - page 17]\nSymbol | Function / cooking mode\n | Oven light\nThis function provides a separate light control to illuminate the\noven cavity without opening the door.\n | Defrosting (fan only)\nOn this setting the fan runs without heat to thaw frozen foods\nrapidly and delicately. It is particularly suitable for defrosting\nfruit and cakes. When using this setting for defrosting easily\nspoilt foods, cook them straight after defrosting, or refrigerate.\n | Fan-forced (fan + centre heating element)\nOn this setting the centre heating element produces the heat\nand the fan distributes it evenly throughout the oven cavity. This\nresults in a fast and economical cooking process, where food\ncan be cooked simultaneously on different shelves, preventing\nthe transmission of smells and tastes from one dish onto another.\nNOTE: This setting is the most suitable for roasting.\n | Radiant grill (top heating element only)\nThis setting utilises the top element, which 

In [9]:
def clean_page(raw_text):
    lines = raw_text.splitlines()
    kept = []
    for line in lines:
        stripped = line.strip()
        # Skip: blank, single number (page num), very short (<4 chars)
        if not stripped:
            continue
        if re.fullmatch(r"\d{1,3}", stripped):
            continue
        if len(stripped) < 4:
            continue
        kept.append(stripped)
    
    cleaned = "\n".join(kept)
    
    # Collapse 3+ consecutive newlines → 2
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    
    # Detect section heading: find last match on this page
    section = "General"
    for match in _HEADING_RE.finditer(cleaned):
        section = match.group(0).strip()[:60]   # cap at 60 chars
    return cleaned, section

In [10]:
chunks = []

In [11]:
step = CHUNK_SIZE - CHUNK_OVERLAP

In [12]:
for page_data in pages:
    cleaned, section = clean_page(page_data["text"])
    words = cleaned.split()

    if len(words) < MIN_CHUNK_WORDS:
        # Page too short to be useful (e.g. title page, blank)
        continue

    for start in range(0, len(words), step):
        chunk_words = words[start : start + CHUNK_SIZE]

        if len(chunk_words) < MIN_CHUNK_WORDS:
            # Tail fragment — too small, skip
            continue

        chunks.append({
            "text":    " ".join(chunk_words),
            "page":    page_data["page"],
            "section": section,
            "type":    "text",
        })

for table_data in tables_out:
    chunks.append({
        "text":    table_data["text"],
        "page":    table_data["page"],
        "section": "Tables / Specifications",
        "type":    "table",
    })
 
print(f"[step4] Created {len(chunks)} chunks "
      f"({sum(1 for c in chunks if c['type']=='text')} text, "
      f"{sum(1 for c in chunks if c['type']=='table')} tables)")

[step4] Created 37 chunks (35 text, 2 tables)


In [13]:
unique_sections = {d['section'] for d in chunks if 'section' in d}

In [14]:
batch_size = 20

In [15]:
all_embeddings = []
texts = [c["text"] for c in chunks]
total_batches = (len(texts) + batch_size - 1) // batch_size

for batch_num, i in enumerate(range(0, len(texts), batch_size), start=1):
    batch = texts[i : i + batch_size]
    print(f"  Embedding batch {batch_num}/{total_batches} ({len(batch)} chunks)...")

    result = genai.embed_content(
        model=EMBED_MODEL,
        content=batch,
        task_type="RETRIEVAL_DOCUMENT",
    )

    # result.embedding is a list of lists (one per text in batch)
    batch_vecs = np.array(result["embedding"], dtype="float32")
    all_embeddings.append(batch_vecs)

embeddings = np.vstack(all_embeddings)
print(f"[step5] Embeddings shape: {embeddings.shape}")  # (num_chunks, 768)

  Embedding batch 1/2 (20 chunks)...
  Embedding batch 2/2 (17 chunks)...
[step5] Embeddings shape: (37, 3072)


In [16]:
def build_faiss_vector(chunks, embeddings, output_dir, appliance_id):
    index_dir = os.path.join(output_dir, appliance_id)
    os.makedirs(index_dir, exist_ok=True)
    
    # Normalise vectors → cosine similarity via inner product
    vecs = embeddings.copy()
    faiss.normalize_L2(vecs)            # in-place L2 normalisation
    
    # Build flat (exact) index — fine for < 50k chunks
    index = faiss.IndexFlatIP(EMBED_DIM)
    index.add(vecs)
    
    # Save index
    index_path = os.path.join(index_dir, "index.faiss")
    faiss.write_index(index, index_path)
    
    # Save chunk metadata (text + page + section)
    chunks_path = os.path.join(index_dir, "chunks.json")
    with open(chunks_path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    
    result = {
        "index_path":  index_path,
        "chunks_path": chunks_path,
        "num_chunks":  index.ntotal,
        "embed_dim":   EMBED_DIM,
    }
    print(f"[step6] Saved {index.ntotal} vectors → {index_dir}")
    return result

In [34]:
def query_index(
    query: str,
    appliance_id: str,
    index_dir: str = "indexes",
    top_k: int = 4,
) -> list[dict]:

    # Load index + chunks
    idx_path    = os.path.join(index_dir, appliance_id, "index.faiss")
    chunks_path = os.path.join(index_dir, appliance_id, "chunks.json")

    index  = faiss.read_index(idx_path)
    with open(chunks_path, encoding="utf-8") as f:
        chunks = json.load(f)

    # Embed the query
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
    )

    # New package returns an object — access via .embeddings[0].values
    q_vec = np.array([result.embeddings[0].values], dtype="float32")
    faiss.normalize_L2(q_vec)

    # Search
    scores, indices = index.search(q_vec, top_k)

    hits = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        hits.append({**chunks[idx], "score": float(score)})

    return hits

In [18]:
result = build_faiss_vector(chunks, embeddings, "indexes", "oven")

[step6] Saved 37 vectors → indexes/oven


In [19]:
query_results = query_index(
            "How to use defrosting?",
            "oven"
        )

In [21]:
for r in query_results:
    print(f"[p.{r['page']} | {r['section']} | score {r['score']:.3f}]")
    print(r['text'][:300])
    print()

[p.17 | Tables / Specifications | score 0.697]
[TABLE - page 17]
Symbol | Function / cooking mode
 | Oven light
This function provides a separate light control to illuminate the
oven cavity without opening the door.
 | Defrosting (fan only)
On this setting the fan runs without heat to thaw frozen foods
rapidly and delicately. It is particularly 

[p.17 | IMPORTANT: | score 0.694]
Instructions (Cont.) Symbol Function / cooking mode Oven light This function provides a separate light control to illuminate the oven cavity without opening the door. Defrosting (fan only) On this setting the fan runs without heat to thaw frozen foods rapidly and delicately. It is particularly suita

[p.23 | General | score 0.666]
Instructions (Cont.) Cooking and baking tips • Before switching on: Before cooking, check that any accessories that are not required are removed from the oven and that only the baking tray and/or the oven rack are positioned on the appropriate shelf support(s). • Preheating: Unless y

In [38]:
def ask_appliance(query, appliance_id, index_dir="indexes", top_k=4):

    # 1. Retrieve relevant chunks (you already have this)
    hits = query_index(query, appliance_id, index_dir, top_k)

    # 2. Build context string from chunks
    context_parts = []
    for i, hit in enumerate(hits, 1):
        context_parts.append(
            f"[Excerpt {i} — Page {hit['page']}, Section: {hit['section']}]\n"
            f"{hit['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # 3. Build the prompt
    prompt = f"""You are a helpful home appliance assistant.
    Answer the user's question using ONLY the manual excerpts below.
    If the answer is not in the excerpts, say "I couldn't find that in the manual."
    Always be specific — mention exact settings, temperatures, or modes from the manual.
    
    MANUAL EXCERPTS:
    {context}
    
    USER QUESTION: {query}
    
    ANSWER:"""

    # 4. Call Gemini LLM
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
    )

    return {
        "answer": response.text,
        "sources": [{"page": h["page"], "section": h["section"], "score": h["score"]} for h in hits]
    }

In [39]:
result = ask_appliance("How do I use the defrosting function?", "oven")
print(result["answer"])
print("\nSources:")
for s in result["sources"]:
    print(f"  p.{s['page']} | {s['section']} | score {s['score']:.3f}")

Based on the manual, here is how to use the defrosting function:

* **Mode/Setting:** Select the **Defrosting (fan only)** setting. 
* **How it works:** On this setting, the fan runs without heat to thaw frozen foods rapidly and delicately.
* **Best used for:** It is particularly suitable for defrosting fruit and cakes. 
* **Important Guidelines:**
  * Always defrost frozen foods thoroughly before cooking (unless packet instructions state to "cook from frozen").
  * When using this setting for easily spoilt foods, cook them straight after defrosting or place them in the refrigerator immediately.

Sources:
  p.17 | Tables / Specifications | score 0.714
  p.17 | IMPORTANT: | score 0.714
  p.23 | General | score 0.666
  p.21 | WARNING! | score 0.655
